# Exploring `sales.csv`

A look at the raw file before writing any cleaning code.

Each section checks one column or one kind of row and gives the row counts used in the README. Rows are counted on the 1,505 order lines (section 4 explains the other two).

In [1]:
import pandas as pd

pd.set_option("display.max_rows", 100)
PATH = "../sales.csv"

## 1. First look: let pandas guess

Reading the file with default settings lets pandas guess each column's type. The guesses are clues.

In [2]:
inferred = pd.read_csv(PATH)
inferred.info()

<class 'pandas.DataFrame'>
RangeIndex: 1507 entries, 0 to 1506
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Order ID    1507 non-null   str  
 1   Region      1506 non-null   str  
 2   Product     1506 non-null   str  
 3   Quantity    1507 non-null   str  
 4   Unit Price  1502 non-null   str  
 5   Discount    544 non-null    str  
 6   Order Date  1506 non-null   str  
dtypes: str(7)
memory usage: 82.5 KB


**Clue:** every column came back as text (`str`), including `Quantity` and `Unit Price`. pandas only does that when
at least one value in the column cannot be read as a number. Sections 7 and 8 find those values.

In [3]:
inferred.isna().sum()

Order ID        0
Region          1
Product         1
Quantity        0
Unit Price      5
Discount      963
Order Date      1
dtype: int64

Missing values per column. `Discount` is mostly empty, which is expected: an empty discount means no discount. The single
missing `Region`, `Product` and `Order Date` turn out to be one and the same row (section 4).

## 2. Read everything as text

For the real checks we read every value as text (`dtype=str`) and keep empty cells as `""`
(`keep_default_na=False`). pandas converts nothing on its own, so we see exactly what is in the file and decide
every conversion ourselves.

In [4]:
raw = pd.read_csv(PATH, dtype=str, keep_default_na=False)
raw.shape

(1507, 7)

## 3. Column headers

`repr()` shows hidden characters such as trailing spaces, which a normal print hides.

In [5]:
[repr(column) for column in raw.columns]

["'Order ID'",
 "'Region'",
 "'Product'",
 "'Quantity '",
 "'Unit Price'",
 "'Discount'",
 "'Order Date'"]

**Found:** `'Quantity '` has a trailing space, so `df["Quantity"]` would fail with a `KeyError`. All seven headers
also contain spaces or capitals, which breaks Part 1's rule (labels are letters and underscores only), so
`add_virtual_column` could not use them. Fix: rename all headers to `snake_case` (`order_id`, `quantity`,
`unit_price`, ...).

## 4. Rows that are not order lines

Exports often carry rows that are not data: a total at the bottom, or a header repeated where two files were
joined. Every real order ID has the shape `SO-` plus five digits, so rows that do not match are suspects.

In [6]:
raw.tail(3)

,Order ID,Region,Product,Quantity,Unit Price,Discount,Order Date
1504,SO-11499,east,Printer paper,20,21.00,,2026-06-24
1505,SO-11500,Central,Monitor stand,20,59.00,59.00,2026-09-26
1506,TOTAL,,,37150,,65924.67,


In [7]:
is_order = raw["Order ID"].str.fullmatch(r"SO-\d{5}")
raw[~is_order]

,Order ID,Region,Product,Quantity,Unit Price,Discount,Order Date
751,Order ID,Region,Product,Quantity,Unit Price,Discount,Order Date
1506,TOTAL,,,37150,,65924.67,


**Found: 2 rows.**
- Line 753 of the file (index 751 below) repeats the header line inside the data (most likely two exports joined into one file).
- The last row is a `TOTAL` line from the source system.

Neither is an order line, and neither can be repaired into one, so both go. The `TOTAL` line is still
useful as a check on our reading of the numbers (section 12).

In [8]:
orders = raw[is_order].copy()
len(orders)

1505

## 5. Region

A column with few distinct values: list them all with `value_counts()`.

In [9]:
orders["Region"].value_counts()

Region
North       311
South       307
Central     295
West        292
East        285
 South        2
EAST          2
south         2
west          1
Central       1
 North        1
north         1
central       1
west          1
south         1
West          1
east          1
Name: count, dtype: int64

There should be a handful of regions, but there are many more spellings. To count the problems, compare each value
with a cleaned version: `strip()` removes outer spaces, `title()` gives `South` from `south` or `SOUTH`.

In [10]:
region = orders["Region"]
stripped = region.str.strip()
normalised = stripped.str.title()

print("extra spaces:  ", (region != stripped).sum())
print("wrong case:    ", (stripped != normalised).sum())
print("rows affected: ", (region != normalised).sum())
print("regions after: ", sorted(normalised.unique()))

extra spaces:   8
wrong case:     10
rows affected:  15
regions after:  ['Central', 'East', 'North', 'South', 'West']


**Found: 15 rows** (8 with extra spaces, 10 with the wrong case, 3 with both). After cleaning there are exactly five
regions. Fix: strip spaces, title case.

## 6. Product

In [11]:
product = orders["Product"]
print("extra spaces:", (product != product.str.strip()).sum())
product.value_counts()

extra spaces: 0


Product
Keyboard         207
Headset          202
Printer paper    196
Notebook pack    189
Desk lamp        187
Monitor stand    183
Office chair     173
Mouse            168
Name: count, dtype: int64

No problems: eight products, spelled consistently.

## 7. Quantity

`pd.to_numeric(..., errors="coerce")` turns anything that is not a number into `NaN`. The important step is to then
**look at the values that failed**, or the coercion hides the problem.

In [12]:
quantity = orders["Quantity "]
quantity_number = pd.to_numeric(quantity, errors="coerce")
quantity[quantity_number.isna()]

250     1 762
734     1 745
1091    1 371
1316    1 427
Name: Quantity , dtype: str

4 rows have a space inside the number (`1 762`). A space is a common thousands separator in European
exports, so `1 762` most likely means 1762. Section 12 confirms it.

In [13]:
quantity_fixed = pd.to_numeric(quantity.str.replace(" ", "", regex=False))
quantity_fixed.describe()

count    1505.000000
mean       24.684385
std        81.492417
min        -5.000000
25%        10.000000
50%        21.000000
75%        31.000000
max      1762.000000
Name: Quantity , dtype: float64

In [14]:
orders[quantity_fixed < 0]

,Order ID,Region,Product,Quantity,Unit Price,Discount,Order Date
421,SO-10421,West,Headset,-2,119.00,,2026-03-30
515,SO-10515,Central,Monitor stand,-4,59.00,,2026-05-10
780,SO-10779,Central,Notebook pack,-5,12.50,,2026-05-02
1014,SO-11011,West,Desk lamp,-3,39.90,,2026-01-03
1293,SO-11289,West,Monitor stand,-3,59.00,,2026-04-12
1450,SO-11445,North,Desk lamp,-2,39.90,,2026-03-08


**6 rows have a negative quantity** (-2 to -5), none with a discount. Most likely returns or corrections.
**Decision:** keep them as returns, so they lower their region's revenue. Their combined revenue is -913.00 (0.04% of
the total). Five of the six have earlier sales of the same product in the same region; SO-11011 (West, desk lamp,
3 January) comes before any such sale. Each negative order ID
appears only once, so a return cannot be linked to the order it returns: a question for the owner of the source
system.

In [15]:
print("largest quantity outside the 4 rows above:", quantity_fixed[quantity_fixed < 1000].max())
orders[quantity_fixed >= 1000]

largest quantity outside the 4 rows above: 40


,Order ID,Region,Product,Quantity,Unit Price,Discount,Order Date
250,SO-10251,North,Printer paper,1 762,21.00,,2026-07-08
734,SO-10734,South,Printer paper,1 745,21.00,,2026-05-29
1091,SO-11088,Central,Monitor stand,1 371,59.00,,2026-07-20
1316,SO-11312,North,Headset,1 427,119.00,,2026-02-04


The four `1 762`-style quantities are also far larger than every other line (the next largest is 40). They are the
only four values of 1,000 or more, which is why only they carry a separator, and the source system's own total counts
them at full size (section 12). **Decision:** keep them and list them in the pipeline's output. Together they are
12.6% of total revenue and 31% of North's.

## 8. Unit price

In [16]:
price = orders["Unit Price"]
price_number = pd.to_numeric(price, errors="coerce")
price[price_number.isna()].value_counts()

Unit Price
          4
39,90     2
59,00     1
29,50     1
249,00    1
89,00     1
21,00     1
Name: count, dtype: int64

**Found: 7 rows** with a decimal comma (`39,90`) and **4 rows** with no price at all. The commas are easy: replace `,` with `.`.

For the missing prices: dropping the rows would lose real orders, so the question is whether they can be fixed.
Check whether a product always has the same price:

In [17]:
price_fixed = pd.to_numeric(price.str.replace(",", ".", regex=False), errors="coerce")
price_fixed.groupby(orders["Product"]).agg(["min", "max", "nunique"])

,min,max,nunique
Product,,,
Desk lamp,39.9,39.9,1
Headset,119.0,119.0,1
Keyboard,89.0,89.0,1
Monitor stand,59.0,59.0,1
Mouse,29.5,29.5,1
Notebook pack,12.5,12.5,1
Office chair,249.0,249.0,1
Printer paper,21.0,21.0,1


Every product has exactly one price across the whole file, so a missing price is filled with that product's
price. The four rows:

In [18]:
orders[price == ""]

,Order ID,Region,Product,Quantity,Unit Price,Discount,Order Date
610,SO-10610,North,Printer paper,40,,,2026-09-04
1102,SO-11099,North,Printer paper,10,,,2026-05-08
1354,SO-11350,North,Keyboard,2,,,2026-01-20
1482,SO-11477,South,Keyboard,37,,164.65,2026-02-25


## 9. Discount

The discount is money taken off the whole line, and an empty value means no discount.

In [19]:
discount = orders["Discount"]
discount_number = pd.to_numeric(discount, errors="coerce")

print("empty:        ", (discount == "").sum())
print("not a number: ", (discount_number.isna() & (discount != "")).sum())
print("negative:     ", (discount_number < 0).sum())

empty:         963
not a number:  0
negative:      0


963 discounts are empty, which means 0. Every other value is a valid, non-negative number.

A sanity check: how big is each discount compared with the line's value?

In [20]:
discount_filled = discount_number.fillna(0)
line_value = quantity_fixed * price_fixed
(discount_filled / line_value)[discount_filled > 0].round(3).value_counts()

0.10    293
0.05    248
Name: count, dtype: int64

Every discount we can check is exactly 5% or 10% of its line. That also tells us something about one of the missing
prices: SO-11477 (Keyboard, 37 pieces, no price) has a discount of 164.65, exactly 5% of 37 × 89.00, the keyboard's
price. That supports filling missing prices from the product (section 8).

In [21]:
row = orders[orders["Order ID"] == "SO-11477"].iloc[0]
float(row["Discount"]) / (int(row["Quantity "]) * 89.00)

0.05

## 10. Order date

In [22]:
dates = pd.to_datetime(orders["Order Date"], format="%Y-%m-%d", errors="coerce")
print("not a valid YYYY-MM-DD date:", dates.isna().sum())
print("range:", dates.min().date(), "to", dates.max().date())

not a valid YYYY-MM-DD date: 0
range: 2026-01-02 to 2026-09-29


No problems: every date is `YYYY-MM-DD`, from 2026-01-02 to 2026-09-29.

## 11. Duplicates

Two questions: are there fully identical rows, and are there order IDs used twice with different content?

In [23]:
print("fully identical rows:         ", orders.duplicated().sum())
print("order IDs used more than once:", orders["Order ID"].duplicated().sum())
orders[orders.duplicated(keep=False)]

fully identical rows:          5
order IDs used more than once: 5


,Order ID,Region,Product,Quantity,Unit Price,Discount,Order Date
304,SO-10305,South,Notebook pack,34,12.50,,2026-05-19
305,SO-10305,South,Notebook pack,34,12.50,,2026-05-19
823,SO-10822,West,Monitor stand,4,59.00,,2026-06-24
824,SO-10822,West,Monitor stand,4,59.00,,2026-06-24
968,SO-10966,Central,Headset,11,119.00,130.90,2026-07-30
969,SO-10966,Central,Headset,11,119.00,130.90,2026-07-30
1243,SO-11240,West,Headset,20,119.00,,2026-07-08
1244,SO-11240,West,Headset,20,119.00,,2026-07-08
1409,SO-11405,North,Desk lamp,27,39.90,,2026-09-05
1410,SO-11405,North,Desk lamp,27,39.90,,2026-09-05


**5 rows** are exact copies of the row directly above them, same order ID and same content. No order ID
is reused with different content. A copy sitting right under its original looks like the export writing a line
twice. **Decision:** remove the copies and keep the first line of each pair.

## 12. Cross-check against the export's own TOTAL line

The `TOTAL` row gives the source system's sums. If our reading of the numbers matches them, the source
system itself confirms that reading.

In [24]:
total = raw[raw["Order ID"] == "TOTAL"].iloc[0]
print("quantity: TOTAL line", total["Quantity "], "| ours", quantity_fixed.sum())
print("discount: TOTAL line", total["Discount"], "| ours", round(discount_filled.sum(), 2))

quantity: TOTAL line 37150 | ours 37150
discount: TOTAL line 65924.67 | ours 65924.67


Both match exactly. This confirms three things:
- `1 762` means 1762 (any other reading would give a different total);
- negative quantities were counted as negatives;
- the duplicates were counted too.

The last point does not settle the duplicates question: the TOTAL line is simply the sum of the lines that were
exported, copies included.

In [25]:
copies = orders.duplicated()
print("quantity without the 5 copies:", quantity_fixed[~copies].sum())

quantity without the 5 copies: 37054


## Summary

| # | Problem | Rows | Handling |
|---|---|---|---|
| 1 | Headers with spaces and capitals (`'Quantity '` has a trailing space) | all 7 columns | rename to `snake_case` |
| 2 | Header line repeated inside the data (line 753) | 1 | remove |
| 3 | `TOTAL` line at the end | 1 | remove; used as a check (section 12) |
| 4 | Region with extra spaces or wrong case | 15 | strip, title case |
| 5 | Quantity with a space as thousands separator (`1 762`) | 4 | space removed; confirmed by TOTAL; kept, listed in the output |
| 6 | Negative quantity | 6 | kept as returns, listed in the output |
| 7 | Unit price with a decimal comma | 7 | `,` to `.` |
| 8 | Unit price missing | 4 | fill with the product's price (one price per product) |
| 9 | Discount empty | 963 | set to 0 (no discount) |
| 10 | Exact duplicate rows | 5 | copies removed |
| 11 | Product, order date | 0 | checked, no problems |